# Search images by text (optional)

> **Heads up.** This is the one heavy notebook. It installs `torch` and `transformers` and
> downloads the CLIP model (~600MB) on first run. The rest of the repo needs none of it.

CLIP embeds images and text into one space, so you can search your postcard images by a text
query. This is an **embedding index**: an index on a column that keeps itself current, the
database's answer to a separate vector database.

In [ ]:
!uv sync --extra clip

In [ ]:
# Your hosted database. Set your org slug (see `pxt org list`).
# This MUST match the [[tool.pixeltable.database]] entry in pyproject.toml.
DB = 'pxt://your-org:champs'  # <-- edit this
print('targeting', DB)

Add an embedding index on the image column. This embeds every image, now and in future. The
first run downloads the model. Then search by a text query: `similarity(string=...)` scores
each image against your words, since CLIP makes text and images comparable.

In [ ]:
import pixeltable as pxt
from pixeltable.functions.huggingface import clip

postcards = pxt.get_table(f'{DB}/postcards')
postcards.add_embedding_index('image',
    embedding=clip.using(model_id='openai/clip-vit-base-patch32'), if_exists='ignore')

sim = postcards.image.similarity(string='a sunny beach')
postcards.order_by(sim, asc=False).select(postcards.caption, score=sim).limit(3).collect()

The database ranked your images by how well each matches the words, with no separate vector
database to run. Change the query and run it again.